# Function Calling: Агент-аналитик с построением графиков

В этом ноутбуке мы научимся создавать **инструмент для построения графиков**, который агент может использовать для визуализации данных из текста.

## Задача

Мы создадим агента, который умеет:
1. Анализировать текстовые данные (например, научную статью)
2. Извлекать числовые данные из текста
3. Строить графики с помощью matplotlib

### Важная особенность

**LLM не может получить изображение обратно!** Когда мы вызываем инструмент для рисования графика, мы можем сохранить изображение, но мы НЕ можем отправить его LLM как результат вызова функции. Поэтому мы используем паттерн **глобального буфера**:

1. Создаём глобальный список `GRAPH_BUFFER`
2. Каждый инструмент сохраняет figure в этот буфер
3. После завершения работы LLM мы отображаем все графики из буфера

Начнём с установки необходимых библиотек:

In [ ]:
%pip install --upgrade openai python-dotenv matplotlib pandas

> **ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

## Авторизация

Для работы с Yandex Cloud нам понадобится `folder_id` (идентификатор каталога) и `api_key` (API-ключ сервисного аккаунта). Мы предполагаем, что эти значения хранятся в файле `.env` в текущей директории.

In [ ]:
!curl -o .env {{url_of_dotenv_file}}

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

---

## Часть 1: Настройка OpenAI клиента

Создадим клиент для работы с Yandex Cloud через OpenAI-совместимый API:

In [ ]:
# Создадим объект `client` для доступа к Responses API

---

## Часть 2: Глобальный буфер для графиков

**Ключевой паттерн!** LLM не может получить изображение напрямую как результат вызова инструмента. Есть несколько архитектурных паттернов, как это сделать:

1. Сохранить изображение на диск и вернуть имя файла
2. Вернуть изображение в [кодировке base64](https://skillbox.ru/media/code/base64) (в этом случае его можно будет [показать внутри Markdown](https://stackoverflow.com/questions/50817518/hard-code-markdown-images))
3. Использовать глобальный буфер для изображений, сохраняя все изображения в него

Мы рекомендуем использовать упрощённый паттерн глобального буфера как наиболее простой:

1. Сохраняем все созданные графики в глобальный буфер
2. Отображаем их после завершения работы агента


In [ ]:
# Определяем переменную для глобального буфера и полезные функции
# для работы с ним: очистка, отображение

---

## Часть 3: Определение инструментов для графиков

Создадим Pydantic-модели для разных типов графиков. Каждая модель:
1. Описывает параметры графика
2. Содержит метод `execute()` для создания графика
3. Сохраняет график в буфер и возвращает текстовое подтверждение

Создайте инструменты для различных типов графиков: обычный график, столбцовая диаграмма, Pie Chart и другие. В графиках предусмотрите возможность называть оси

In [ ]:
# Определите набор инструментов, унаследованных от pydantic.BaseModel

---

## Часть 4: Класс Agent для построения графиков

Используем класс агента `Agent`, аналогичный тому, что использовался в примерах.

In [ ]:
# Определите класс `Agent`. Можете взять код из лекционных примеров.

---

## Часть 5: Создание агента-аналитика

Теперь создадим агента с инструкцией для анализа текстов и построения графиков.

In [ ]:
grapher_instruction = """
Ты — агент-аналитик для визуализации данных. Твоя задача:

1. Внимательно анализировать предоставленный текст
2. Извлекать числовые данные, статистику, сравнения
3. Выбирать подходящий тип графика для визуализации
4. Строить информативные графики с понятными подписями

Типы графиков:
- DrawBarChart: для сравнения категорий (например, модели, методы)
- DrawLineChart: для трендов и временных рядов (например, по годам)
- DrawPieChart: для долей от целого (например, распределение)
- DrawGroupedBarChart: для сравнения нескольких серий данных

Правила:
- Всегда используй понятные заголовки на русском языке
- Добавляй подписи осей
- Если данных много, выбирай самые важные для визуализации
- Объясняй, что показывает каждый график
"""

# Создаём агента
grapher = GrapherAgent(
    instruction=grapher_instruction,
    tools=[...]
)

print("✅ Агент-аналитик готов к работе")

---

## Часть 6: Загрузка статьи "Attention Is All You Need"

Загрузим текст статьи для анализа.

In [1]:
!curl -o article-attention.txt https://raw.githubusercontent.com/yandex-ai-studio/ai-studio-course/refs/heads/main/data/article-attention.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
 10 38203   10  4134    0     0   2226      0  0:00:17  0:00:01  0:00:16  2259
100 38203  100 38203    0     0  19896      0  0:00:01  0:00:01 --:--:-- 20170


In [ ]:
# Загружаем статью
with open("article-attention.txt", "r", encoding="utf-8") as f:
    article_text = f.read()

print(article_text[:500])

---

## Часть 7: Анализ ссылок по годам

Попросим агента проанализировать список литературы и построить график распределения цитируемых публикаций по годам.

In [ ]:
# Очищаем буфер перед новым анализом
clear_graph_buffer()

# Извлекаем секцию References из статьи
# Выделите весь текст, расположенный после текста "References"
references_text = ""

prompt = f"""
Напишите промпт

Текст раздела References:
{references_text}
"""

print("🔍 Анализируем ссылки...")
result = grapher(prompt)
print(f"\n💬 Результат: {result}")

In [ ]:
# HINT: Отображаем графики ПОСЛЕ завершения работы агента
# Это ключевой момент — LLM не видит эти графики,
# они сохраняются в буфере и отображаются здесь
display_all_graphs()

---

## Часть 8: Сравнение BLEU Score моделей

Извлечём данные из Table 2 статьи и визуализируем сравнение моделей машинного перевода.

In [ ]:
clear_graph_buffer()

# Извлекаем раздел с результатами
results_section = article_text[article_text.find("6Results"):article_text.find("7Conclusion")]

prompt = f"""
Проанализируй раздел Results из статьи "Attention Is All You Need" и построй графики:

1. Сгруппированную столбчатую диаграмму BLEU Score для разных моделей 
   (ByteNet, GNMT+RL, ConvS2S, Transformer и т.д.) на задачах EN-DE и EN-FR.
   Если для какой-то модели нет данных по одной из задач, пропусти её.

Текст раздела Results:
{results_section}

Извлеки BLEU scores из Table 2 и визуализируй их.
"""

print("🔍 Анализируем результаты моделей...")
result = grapher(prompt)
print(f"\n💬 Результат: {result}")

In [ ]:
display_all_graphs()

---

## Часть 9: Анализ авторов и организаций

Построим круговую диаграмму распределения авторов по организациям.

In [ ]:
clear_graph_buffer()

# Берём информацию об авторах (первые строки статьи)
authors_section = article_text[:1500]

prompt = f"""
Проанализируй информацию об авторах статьи и построй круговую диаграмму,
показывающую распределение авторов по организациям (Google Brain, Google Research, 
University of Toronto и т.д.).

Текст с информацией об авторах:
{authors_section}

Подсчитай количество авторов от каждой организации и визуализируй.
"""

print("🔍 Анализируем авторов...")
result = grapher(prompt)
print(f"\n💬 Агент: {result}")

In [ ]:
display_all_graphs()

---

## Выводы

В этом ноутбуке мы научились:

1. **Создавать инструменты для визуализации** — использовать Pydantic-модели для описания параметров графиков

2. **Решать проблему "LLM не видит картинки"** — использовать паттерн глобального буфера для сохранения графиков и отображения их после работы агента

3. **Извлекать данные из текста** — LLM способен находить и структурировать числовые данные из неструктурированного текста

4. **Выбирать подходящий тип визуализации** — агент сам решает, какой график лучше подходит для конкретных данных

### Практические советы

- Всегда очищайте буфер `clear_graph_buffer()` перед новым анализом
- Инструменты должны возвращать текстовое описание результата
- Используйте понятные имена и описания для функций
- Давайте агенту достаточно контекста в промпте